# 뉴스 기사 정보 추출 (LLM)

`news_filter.ipynb`를 통과한 기사에서 **개체(기업·제품·원자재·정책 등), 관계, 사건·지표**를 LLM으로 뽑는다. 항목과 원칙은 `graphDB.md` 5장(5.1 공통, 5.3 뉴스)을 따른다.

- **요청 단위**: 기사 하나를 요청 한 번으로 보낸다(5.1).
- **실행 수단**(`graphDB.md` 4.4, 1절 `RUNNER`):
  - `'codex'`: 지시문 다듬기. 구독 Codex로 시범 기사를 돌리고, 결과 분석과 지시문 수정은 Claude Code가 한다. **시험에는 API를 쓰지 않는다.**
  - `'batch'`: 쌓인 기사 전체 추출. OpenAI 배치 API(24시간 안 처리, 가격 50%)로 보낸다. 제출한 뒤 노트북을 다시 실행하면 결과를 받아 기사별로 나눠 저장한다(5-1절).
- **모델**: `gpt-6.1-terra`(2026-11-12 결정, 5.3 모델 비교).
- **시범 기사는 고정한다.** 처음 실행할 때 매체 업종별로 골라 `sample.csv`에 저장하고, 지시문 버전이 바뀌어도 같은 기사로 비교한다.
- **원본 결과를 보관한다**(5.4). 응답 전체를 기사별 파일로 남기고 고치지 않는다.
- **실행은 꺼 두었다.** 1절 `RUN = True`로 바꿔야 실행된다. Codex는 구독 사용량, 배치는 API 크레딧을 쓰므로 건수와 예상 비용을 보고 돌린다.
- 같은 지시문 버전·모델로 성공한 기사는 실행 수단과 관계없이 다시 돌리지 않는다. 크레딧 부족 등으로 실패한 기사는 다음 실행 때 다시 대상이 된다.

**입력**

| 파일 | 내용 |
|---|---|
| `source/news/filter/article_filter.csv` | 기사별 사용 여부(`use`: body / summary / exclude) (`news_filter`) |
| `source/news/snapshot_*/news.sqlite3` | 기사 원문(가장 최근 스냅샷) |
| `prompts/news_버전.md`, `prompts/news_버전.schema.json` | 지시문과 결과 JSON 스키마 |

**출력** (모두 `source/news/extract/` 아래)

| 파일 | 내용 |
|---|---|
| `sample.csv` | 시범 기사 목록(고정) |
| `raw/지시문버전/실행도구-모델/기사ID앞12자_실행시각.json` | LLM 응답 전체. 고치지 않는다. Codex는 `codex-모델/`, 배치는 `api-모델/` |
| `batches/`, `batches.csv` | 배치로 보낸 요청 파일과 배치 기록(배치 ID, 상태, 기사 수, 결과를 받았는지) |
| `runs.csv` | 실행 기록: 기사, 입력 해시, 지시문 버전, 실행 도구, 모델, 시각, 상태, 토큰, 비용(배치) |
| `check_지시문버전_실행도구-모델.csv` | 기사별 점검 결과 |

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `news_filter` → **`news_extractor`** → `news_cleaner` |
| 외부 요청·비용 | 있음. `RUNNER='batch'`는 OpenAI 배치 API(크레딧), `'codex'`는 Codex 구독. 실행 전에 4절의 예상 건수·비용을 보고 승인받는다 |
| 다시 실행 | 된다. 같은 지시문·모델로 성공한 기사는 건너뛰고, 배치는 실행할 때마다 결과를 받고 남은 기사를 새로 제출한다 |
| 멈추는 경우 | `RUN=False`(기본)면 예상만 보여 주고 보내지 않는다. Codex 사용량 한도, 배치 제출 실패(크레딧 부족 등)에서 멈춘다 |
| 처리 순서 | 설정 → 입력 글 만들기 → 시범 기사 → 대상 고르기·예상 비용 → 실행(Codex) 또는 배치 제출·결과 받기 → 점검 → 기사 하나 보기 |

## 1. 설정

In [ ]:
# ── 설정: 지시문 버전, 실행 수단·모델, 대상, 배치 한도, 가격 ──
import datetime as dt
import hashlib
import html
import json
import os
import re
import sqlite3
import subprocess
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import jsonschema
import pandas as pd

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
NEWS = ROOT / 'source' / 'news'
EXTRACT = NEWS / 'extract'
EXTRACT.mkdir(parents=True, exist_ok=True)
RUNS_PATH = EXTRACT / 'runs.csv'
SAMPLE_PATH = EXTRACT / 'sample.csv'

PROMPT_VERSION = 'v5'
PROMPT_PATH = ROOT / 'prompts' / f'news_{PROMPT_VERSION}.md'
SCHEMA_PATH = ROOT / 'prompts' / f'news_{PROMPT_VERSION}.schema.json'
SCHEMA = json.loads(SCHEMA_PATH.read_text(encoding='utf-8'))

# ── 실행 설정 ──
RUNNER = 'batch'          # 'codex'(지시문 다듬기, 구독) 또는 'batch'(전체 추출, OpenAI 배치 API)
MODEL = 'gpt-6.1-terra'   # 뉴스 모델(2026-11-12 결정). Luna는 비교 시험에만 썼다
EFFORT = 'medium'         # 추론 강도: low, medium, high
PARALLEL = 3              # 동시에 돌릴 수
TIMEOUT_SEC = 600         # 기사 하나에 기다리는 최대 시간

# 대상: 'test1'(시범 첫 기사), 'sample'(시범 기사 전체), 'all'(쓰는 기사 전체), 또는 시범 기사 번호 목록(sample.csv 순서, 예: [3, 7, 10])
TARGET = 'all'

# 배치 대기열에 한 번에 넣을 입력 토큰. 요청 토큰은 어림값이라 85%까지만 채운다
# Build 등급은 300만이었다. Grow 등급(2026-11-12 새 계정)은 2,000만으로 올려 쌓인 기사 전체를 배치 하나로 보낸다
BATCH_QUEUE_TOKENS = 20_000_000
BATCH_FILL = 0.85

# '끝남'으로 칠 결과의 실행 수단. 기본은 지금 RUNNER와 같은 수단만 친다.
# 전체 재추출(batch)은 시범 기사에 Codex v5 결과가 있어도 API로 다시 뽑아 한 수단으로 맞춘다
DONE_RUNNERS = [RUNNER]

RUN = False               # True로 바꿔야 실행한다

if RUNNER == 'batch':
    # API는 구조화 출력(엄격 모드)으로 결과 형식을 정한다
    PROMPT = PROMPT_PATH.read_text(encoding='utf-8')
    RUN_NAME = f'api-{MODEL}'
else:
    # Codex는 --output-schema로 결과 형식을 정한다
    PROMPT = (PROMPT_PATH.read_text(encoding='utf-8')
              + '\n\n## 결과 형식\n\n결과는 주어진 출력 스키마에 맞는 JSON 하나다. 파일을 읽거나 명령을 실행하지 말고, 아래 입력만 보고 답한다.')
    RUN_NAME = f'codex-{MODEL}'

# OpenAI API 가격(100만 토큰당 입력, 캐시 입력, 출력. 2026-11-12 공식 가격표). 배치는 모두 50%
PRICES = {'gpt-6.1-terra': (1.00, 0.05, 5.00), 'gpt-6-luna': (0.10, 0.01, 0.50)}


def api_cost(model, input_tokens, cached_tokens, output_tokens, batch=False):
    """토큰 수로 OpenAI 비용(달러)을 계산한다. 배치는 절반."""
    p_in, p_cache, p_out = PRICES[model]
    usd = ((input_tokens - cached_tokens) * p_in + cached_tokens * p_cache + output_tokens * p_out) / 1e6
    return usd / 2 if batch else usd


def env_value(name):
    """환경변수 → 프로젝트 루트 .env → graph_db/.env 순서로 값을 찾는다. 값은 화면에 쓰지 않는다."""
    if os.environ.get(name):
        return os.environ[name]
    for env_file in (ROOT.parent / '.env', ROOT / '.env'):
        if env_file.exists():
            for line in env_file.read_text(encoding='utf-8-sig').splitlines():
                key, _, value = line.partition('=')
                if key.strip() == name and value.strip():
                    return value.strip().strip('"\'')
    return None


RAW = EXTRACT / 'raw' / PROMPT_VERSION / RUN_NAME
RAW.mkdir(parents=True, exist_ok=True)
SNAPSHOT = sorted(NEWS.glob('snapshot_*/news.sqlite3'))[-1]
print('실행 도구', RUNNER, '/ 모델', MODEL, '/ effort', EFFORT, '/ 지시문', f'{len(PROMPT):,}자',
      '/ 기사 원문', SNAPSHOT.relative_to(ROOT), '/ 결과', RAW.relative_to(ROOT))

## 2. 기사 입력 글 만들기

거르기 결과에서 쓰는 기사(`use`가 body 또는 summary)만 읽는다. 입력 글은 제목·매체·발행 시각을 맨 위에 두고, 그 아래에 본문(`body`) 또는 요약(`summary`, HTML 태그를 지운 글)을 붙인다.

In [ ]:
# ── 입력 글: 거르기를 통과한 기사의 제목·매체·발행 시각 + 본문(또는 요약) ──
def strip_html(s):
    """요약 HTML에서 태그를 지우고 글만 남긴다."""
    s = re.sub(r'<(script|style)[^>]*>.*?</\1>', ' ', s or '', flags=re.S | re.I)
    s = html.unescape(re.sub(r'<[^>]+>', ' ', s))
    return re.sub(r'[ \t]+', ' ', s).strip()


flt = pd.read_csv(NEWS / 'filter' / 'article_filter.csv', dtype=str).fillna('')
flt = flt[flt['use'].isin(['body', 'summary'])]
with sqlite3.connect(SNAPSHOT) as con:
    art = pd.read_sql('select article_id, source, title, published_utc, summary_html, body_text from articles', con)
    feeds = pd.read_sql('select af.article_id, f.sector, f.feed_name from article_feeds af join feeds f using(feed_id)', con)
art = flt[['article_id', 'use']].merge(art, on='article_id')
# 기사가 실린 피드의 업종(여러 피드면 |로 잇는다)
sector_of = feeds.assign(sector=feeds['sector'].fillna('').replace('', '일반')).groupby('article_id')['sector'].agg(
    lambda s: '|'.join(sorted(set(s))))
art['feed_sector'] = art['article_id'].map(sector_of).fillna('일반')


def input_text(r):
    """LLM에 보낼 기사 입력 글을 만든다. use=body면 본문, summary면 RSS 요약."""
    body = r['body_text'] if r['use'] == 'body' else strip_html(r['summary_html'])
    return f"Title: {r['title']}\nSource: {r['source']}\nPublished: {r['published_utc']}\n\n{body.strip()}"


art['text'] = art.apply(input_text, axis=1)
art['chars'] = art['text'].str.len()
TEXT = dict(zip(art['article_id'], art['text']))
print(f'쓰는 기사 {len(art):,}건 (본문 {(art["use"] == "body").sum():,}, 요약 {(art["use"] == "summary").sum():,}) / '
      f'입력 글자 중간값 {int(art["chars"].median()):,}, 90% {int(art["chars"].quantile(.9)):,}')

## 3. 시범 기사 고르기

`sample.csv`가 없을 때 한 번만 만든다(이후에는 같은 기사로 지시문 버전을 비교한다).

- 매체 업종(`feeds.sector`, 업종이 없는 일반 매체는 "일반")마다 본문 기사 1건(일반은 4건)을 고른다. 길이 1,500~8,000자에서 무작위(고정 시드).
- 유형을 넓히려고 긴 본문 2건(1.2만 자 이상), 요약만 있는 기사 2건, **국내 기업(삼성, SK hynix, 현대 등)이 나오는 기사 5건**을 더한다.
- 추가 묶음(`group`): 관세·무역, 인수합병, 공급 차질, 원자재 가격, 보도자료형 각 2건, 국내 기업 5건 더, 짧은 본문(400~1,200자) 2건. 앞에서 고른 기사는 그대로 두고 뒤에 붙인다.
- **문제 유형**(`group`이 "문제 유형: …"인 18건): v4 배치 결과를 검토하며 고른 기사다. "기타" 관계가 많았던 기사, 공급 관계가 빠졌던 기사, 증권사 투자의견, 소송·특허, 보조금·대출, 법안·정책 입장. 기사 ID를 `sample.csv`에 직접 적어 두었다(이 칸의 규칙으로 고르지 않는다).

In [ ]:
# ── 시범 기사: 지시문 버전을 같은 기사로 비교하려고 한 번 고정한다 ──
KOREA_NAMES = (r'Samsung|SK hynix|SK Hynix|Hyundai|LG Energy|LG Electronics|LG Chem|\bKia\b|POSCO|Posco|Hanwha|Doosan|'
               r'Korea Zinc|Celltrion|Naver|Kakao|Lotte|SK On|Korean Air')
if not SAMPLE_PATH.exists():
    pool = art[(art['use'] == 'body') & art['chars'].between(1500, 8000)]
    picks = []
    for sector, g in pool.groupby('feed_sector'):
        picks.append(g.sample(min(len(g), 4 if sector == '일반' else 1), random_state=7))
    picks.append(art[(art['use'] == 'body') & (art['chars'] >= 12000)].sample(2, random_state=7))
    picks.append(art[art['use'] == 'summary'].sample(2, random_state=7))
    korean = art[(art['use'] == 'body') & art['text'].str.contains(KOREA_NAMES) & art['chars'].between(1500, 10000)]
    picks.append(korean.sample(5, random_state=7))
    sample = pd.concat(picks).drop_duplicates('article_id')
    sample[['article_id', 'feed_sector', 'source', 'use', 'chars', 'title']].to_csv(SAMPLE_PATH, index=False, encoding='utf-8-sig')
sample = pd.read_csv(SAMPLE_PATH, dtype={'article_id': str})
if 'group' not in sample.columns:
    sample['group'] = '기본'

# 기사 유형을 넓히는 추가 묶음(제목이나 본문 앞부분의 표현으로 고른다). 없는 묶음만 뒤에 붙이므로 앞의 기사는 바뀌지 않는다
EXTRA = [('관세·무역', r'\btariff', 2), ('인수합병', r'\bacquir|\bmerger\b', 2), ('공급 차질', r'shortage|outage|\bstrike\b|disrupt', 2),
         ('원자재 가격', r'oil price|crude price|copper price|LNG price|iron ore price', 2), ('보도자료형', r'announced today|today announced', 2),
         ('국내 기업 추가', KOREA_NAMES, 5), ('짧은 본문', None, 2)]
added = False
for group, pattern, n in EXTRA:
    if group in set(sample['group']):
        continue
    rest = art[~art['article_id'].isin(sample['article_id'])]
    if pattern is None:
        cand = rest[(rest['use'] == 'body') & rest['chars'].between(400, 1200)]
    else:
        head = rest['title'] + ' ' + rest['body_text'].fillna('').str[:600]
        cand = rest[(rest['use'] == 'body') & head.str.contains(pattern, case=False, regex=True) & rest['chars'].between(800, 10000)]
    pick = cand.sample(min(n, len(cand)), random_state=7).assign(group=group)
    sample = pd.concat([sample, pick[['article_id', 'feed_sector', 'source', 'use', 'chars', 'title', 'group']]], ignore_index=True)
    added = True
if added:
    sample.to_csv(SAMPLE_PATH, index=False, encoding='utf-8-sig')
print(f'시범 기사 {len(sample)}건, 입력 {sample["chars"].sum():,}자 / 묶음별', sample['group'].value_counts().to_dict())
sample[['group', 'feed_sector', 'source', 'use', 'chars', 'title']]

## 4. 대상 고르기

입력 글의 해시로 입력이 바뀌었는지 안다. 같은 지시문 버전·모델로 성공한 기사는 실행 수단(Codex, 배치)과 관계없이 건너뛴다. 배치로 제출해 결과를 기다리는 기사도 건너뛴다.

**요청 토큰 어림**(배치 대기열 계산과 예상 비용): 시범 57건의 실제 값으로 맞춘 식이다. 입력은 지시문·스키마 약 5,600토큰 + 0.19 × 글자, 출력은 1,134 + 0.10 × 글자. 지시문·스키마 부분은 기사마다 같아서 캐시 가격으로 계산될 수 있다(DART 배치에서 입력의 약 64%가 캐시). 예상 비용은 캐시 없음과 지시문 캐시 두 경우로 보여 준다.

In [ ]:
# ── 대상 고르기: 이미 끝난 기사·배치를 기다리는 기사를 빼고, 예상 토큰·비용을 보여 준다 ──
def text_hash(text):
    """입력 글의 짧은 해시. 원문이 바뀌면 다시 돌린다."""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]


PROMPT_HASH = text_hash(PROMPT + RUN_NAME + EFFORT)
RUN_COLUMNS = ['article_id', 'input_hash', 'prompt_version', 'prompt_hash', 'runner', 'model', 'effort', 'run_utc', 'raw_file',
               'status', 'error', 'duration_s', 'input_tokens', 'cache_read_tokens', 'output_tokens', 'reasoning_tokens', 'cost_usd']
runs = pd.read_csv(RUNS_PATH, dtype=str) if RUNS_PATH.exists() else pd.DataFrame(columns=RUN_COLUMNS)
if RUNS_PATH.exists() and list(runs.columns) != RUN_COLUMNS:
    # 열이 바뀌었으면(예: 비용 열 추가) 새 열 구성으로 옮겨 쓴다. 그래야 한 줄씩 붙일 때 열이 맞는다
    runs.reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, index=False, encoding='utf-8-sig')
runs = runs.reindex(columns=RUN_COLUMNS)

if TARGET == 'test1':
    wanted = list(sample['article_id'][:1])
elif TARGET == 'sample':
    wanted = list(sample['article_id'])
elif TARGET == 'all':
    wanted = list(art['article_id'])
else:
    wanted = [sample['article_id'][i] for i in TARGET]

# 토큰 어림값: 지시문 + 글자 수 비례(시범 결과로 맞춘 값)
PROMPT_TOKENS = 5_600
est_in = lambda a: PROMPT_TOKENS + 0.19 * len(TEXT[a])
est_out = lambda a: 1_134 + 0.10 * len(TEXT[a])

BATCH_DIR = EXTRACT / 'batches'
BATCH_LOG = EXTRACT / 'batches.csv'


def batch_log():
    """배치 기록(batches.csv)을 읽는다."""
    return pd.read_csv(BATCH_LOG, dtype=str).fillna('') if BATCH_LOG.exists() else pd.DataFrame(
        columns=['batch_id', 'created_utc', 'input_file', 'n', 'est_tokens', 'prompt_version', 'prompt_hash', 'model', 'effort', 'status', 'collected'])


def waiting_ids():
    """제출했지만 아직 결과를 받지 않은 배치의 기사."""
    ids = set()
    for b in batch_log().query("collected != 'yes'").itertuples():
        ids |= {json.loads(l)['custom_id'].split('|')[0] for l in (BATCH_DIR / b.input_file).read_text(encoding='utf-8').splitlines() if l.strip()}
    return ids


def remaining():
    """아직 돌리지 않은 대상 기사와 배치 결과를 기다리는 기사를 돌려준다."""
    done = set(runs.loc[(runs['status'] == 'ok') & (runs['prompt_version'] == PROMPT_VERSION) & (runs['model'] == MODEL)
                        & runs['runner'].isin(DONE_RUNNERS), ['article_id', 'input_hash']].itertuples(index=False, name=None))
    waiting = waiting_ids()
    return [a for a in wanted if a not in waiting and (a, text_hash(TEXT[a])) not in done], waiting


todo, waiting = remaining()
print(f'대상 {len(wanted):,}건 / 이미 끝남 {len(wanted) - len(todo) - len(waiting & set(wanted)):,}건 / 배치 결과 기다림 {len(waiting & set(wanted)):,}건'
      f' / 이번에 돌릴 것 {len(todo):,}건, 입력 {sum(len(TEXT[a]) for a in todo):,}자')
if RUNNER == 'batch' and todo:
    tin, tout = sum(est_in(a) for a in todo), sum(est_out(a) for a in todo)
    lo = api_cost(MODEL, tin, PROMPT_TOKENS * len(todo), tout, batch=True)
    hi = api_cost(MODEL, tin, 0, tout, batch=True)
    print(f'예상(배치, {MODEL}): 입력 약 {tin / 1e6:.1f}백만 토큰, 출력 약 {tout / 1e6:.1f}백만 토큰 → 약 ${lo:.1f}(지시문 캐시) ~ ${hi:.1f}(캐시 없음)')

## 5. 실행 (Codex)

`RUNNER = 'codex'`일 때만 돈다.

기사 하나마다 `codex exec`를 새로 띄운다. 옵션은 `dart_business_extractor`와 같다(빈 임시 폴더, 읽기 전용, 세션 기록·사용자 설정 끔, `--output-schema`로 결과 형식 고정, 모델·추론 강도 고정). 환경변수 `OPENAI_API_KEY`는 지워서 구독으로만 돈다.

응답은 성공·실패와 관계없이 `raw/`에 그대로 저장하고, 기사 하나가 끝날 때마다 `runs.csv`에 바로 한 줄 남긴다. 사용량 한도 오류가 나면 남은 기사는 돌리지 않는다.

In [ ]:
# ── 실행 도구: Codex 실행(RUNNER='codex')과 배치 요청·결과 해석에 쓰는 함수 ──
RUN_CWD = Path(tempfile.gettempdir()) / 'graphdb_llm_run'
RUN_CWD.mkdir(exist_ok=True)
# Codex에 API 키가 넘어가지 않게 뺀다(구독으로만 돈다)
ENV = {k: v for k, v in os.environ.items() if k not in ('ANTHROPIC_API_KEY', 'ANTHROPIC_AUTH_TOKEN', 'OPENAI_API_KEY')}
LIMIT_RE = re.compile(r'usage limit|rate limit|limit reached|quota|too many requests|429', re.I)
VALIDATOR = jsonschema.Draft202012Validator(SCHEMA)


def command(out_file):
    """기사 하나를 돌리는 codex exec 명령(읽기 전용, 사용자 설정 무시, 출력 스키마 지정)."""
    return ['codex', 'exec', '--json', '--ephemeral', '--skip-git-repo-check', '-s', 'read-only', '--ignore-user-config',
            '--ignore-rules', '-C', str(RUN_CWD), '--output-schema', str(SCHEMA_PATH), '-o', str(out_file),
            '-c', f'model_reasoning_effort="{EFFORT}"', '-m', MODEL, '-']


def parse_result(resp):
    """원본 응답에서 결과 JSON을 꺼낸다(Codex: 최종 메시지, 배치 한 줄: 응답 본문의 output)."""
    if 'custom_id' in resp:
        resp = (resp.get('response') or {}).get('body') or {}
    if isinstance(resp.get('output'), list):
        text = next((c.get('text') or '' for item in resp['output'] if item.get('type') == 'message'
                     for c in item.get('content') or [] if c.get('type') == 'output_text'), '')
    else:
        text = resp.get('last_message') or ''
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    return json.loads(text[text.find('{'):])


mask = lambda s: re.sub(r'org-[A-Za-z0-9]{6,}', 'org-***', re.sub(r'sk-[A-Za-z0-9_\-]{6,}', 'sk-***', str(s)))   # 키·조직 ID를 가린다
_client = None


def openai_client():
    """OpenAI 클라이언트를 한 번만 만든다. 키는 .env에서 읽는다."""
    global _client
    if _client is None:
        import openai
        key = env_value('OPENAI_API_KEY')
        assert key, '.env에 OPENAI_API_KEY가 없다.'
        _client = openai.OpenAI(api_key=key, max_retries=3)
    return _client


def api_body(text):
    """배치 요청 하나의 본문(Responses API)."""
    return dict(model=MODEL, instructions=PROMPT, input=text, reasoning={'effort': EFFORT}, store=False,
                text={'format': {'type': 'json_schema', 'name': 'news', 'schema': SCHEMA, 'strict': True}})


def finish(row, body):
    """배치 응답 본문의 토큰·비용을 적고 결과를 검사한다."""
    u = body.get('usage') or {}
    cached = (u.get('input_tokens_details') or {}).get('cached_tokens') or 0
    row.update(input_tokens=u.get('input_tokens'), cache_read_tokens=cached, output_tokens=u.get('output_tokens'),
               reasoning_tokens=(u.get('output_tokens_details') or {}).get('reasoning_tokens'))
    if u.get('input_tokens') is not None:
        row['cost_usd'] = round(api_cost(MODEL, u['input_tokens'], cached, u.get('output_tokens') or 0, batch=True), 5)
    if body.get('status') != 'completed':
        row.update(status='error', error=mask(json.dumps(body.get('incomplete_details') or body.get('error') or body.get('status')))[:300])
        return row
    try:
        errs = [e.message for e in VALIDATOR.iter_errors(parse_result(body))]
        row.update(status='ok' if not errs else 'bad_schema', error='; '.join(errs)[:300])
    except (json.JSONDecodeError, TypeError) as e:
        row.update(status='bad_json', error=str(e)[:300])
    return row


def run_one(aid):
    """Codex로 기사 하나를 돌리고 원본 응답을 저장한 뒤 실행 기록 한 줄을 돌려준다."""
    text = TEXT[aid]
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
    raw_file = f"{aid[:12]}_{now.strftime('%Y%m%dT%H%M%S')}.json"
    out_file = RUN_CWD / f'news_{aid[:12]}_{now.strftime("%H%M%S")}.out'
    row = dict(article_id=aid, input_hash=text_hash(text), prompt_version=PROMPT_VERSION, prompt_hash=PROMPT_HASH, runner=RUNNER,
               model=MODEL, effort=EFFORT, run_utc=now.isoformat(), raw_file=raw_file, status='', error='')
    t0 = time.time()
    try:
        proc = subprocess.run(command(out_file), input=PROMPT + '\n\n---\n\n# 입력\n\n' + text, capture_output=True, text=True,
                              timeout=TIMEOUT_SEC, cwd=RUN_CWD, env=ENV)
    except subprocess.TimeoutExpired:
        return row | dict(status='timeout', duration_s=round(time.time() - t0, 1))
    row['duration_s'] = round(time.time() - t0, 1)
    events = []
    for line in proc.stdout.splitlines():
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            events.append({'type': 'text', 'text': line})
    last = out_file.read_text(encoding='utf-8') if out_file.exists() else ''
    out_file.unlink(missing_ok=True)
    usage = next((e['usage'] for e in reversed(events) if e.get('type') == 'turn.completed' and isinstance(e.get('usage'), dict)), {})
    errors = [e for e in events if e.get('type') in ('error', 'turn.failed')]
    resp = {'runner': RUNNER, 'model': MODEL, 'article_id': aid, 'returncode': proc.returncode, 'usage': usage,
            'last_message': last, 'events': events, 'stderr': proc.stderr[-3000:]}
    (RAW / raw_file).write_text(json.dumps(resp, ensure_ascii=False, indent=1), encoding='utf-8')
    row.update(input_tokens=usage.get('input_tokens'), cache_read_tokens=usage.get('cached_input_tokens'),
               output_tokens=usage.get('output_tokens'), reasoning_tokens=usage.get('reasoning_output_tokens'))
    if errors or proc.returncode != 0 or not last:
        msg = ' '.join([proc.stderr, json.dumps(errors, ensure_ascii=False)])
        return row | dict(status='limit' if LIMIT_RE.search(msg) else 'error', error=msg.strip()[:300])
    try:
        errs = [e.message for e in VALIDATOR.iter_errors(parse_result(resp))]
        row.update(status='ok' if not errs else 'bad_schema', error='; '.join(errs)[:300])
    except (json.JSONDecodeError, TypeError) as e:
        row.update(status='bad_json', error=str(e)[:300])
    return row


# 동시 PARALLEL개씩 돌리고, 하나 끝날 때마다 실행 기록에 붙인다. 한도에 걸리면 새로 시작하지 않는다
if RUN and todo and RUNNER == 'codex':
    print(f'{len(todo)}건 실행 시작 (동시 {PARALLEL}개)')
    stop = False
    with ThreadPoolExecutor(PARALLEL) as pool:
        pending = {pool.submit(run_one, a): a for a in todo[:PARALLEL]}
        queue = list(todo[PARALLEL:])
        while pending:
            for fut in as_completed(list(pending)):
                a = pending.pop(fut)
                row = fut.result()
                pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(
                    RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
                print(f"  {a[:12]} {row['status']:8} {row.get('duration_s', '')}초 입력 {row.get('input_tokens')} 출력 {row.get('output_tokens')}")
                stop = stop or row['status'] == 'limit'
                if queue and not stop:
                    nxt = queue.pop(0)
                    pending[pool.submit(run_one, nxt)] = nxt
                break
    if stop:
        print('사용량 한도에 걸려 멈췄다. 한도가 풀린 뒤 다시 실행하면 남은 기사부터 한다.')
    runs = pd.read_csv(RUNS_PATH, dtype=str).reindex(columns=RUN_COLUMNS)
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

## 5-1. 배치 제출과 결과 받기

`RUNNER = 'batch'`일 때만 쓴다. 노트북을 실행할 때마다 아래 순서로 한다.

1. **결과 받기**(요금 없음): 기다리는 배치의 상태를 확인하고, 끝났으면 결과 파일과 오류 파일을 받아 기사별 원본 결과(`raw/<지시문 버전>/api-모델/기사ID앞12자_시각_batch.json`)로 나누고 `runs.csv`에 남긴다. 실제 토큰과 비용(`cost_usd`)도 적는다.
2. **남은 기사 다시 계산**: 방금 받은 결과까지 반영한다.
3. **고르기**: 대기열에 남은 자리(`BATCH_QUEUE_TOKENS`의 85%에서 기다리는 배치의 요청 토큰을 뺀 양)만큼 기사를 고른다.
4. **제출**(`RUN = True`일 때): 고른 기사로 요청 파일(`batches/이름.jsonl`)을 만들어 올리고 배치를 만든다. 배치 ID는 `batches.csv`에 남긴다.

- 기사 2,400여 건은 대기열 한도 때문에 6번 정도에 나눠 제출된다. 앞 배치가 끝나면 다시 실행해 다음을 낸다.
- **크레딧이 떨어지면** 그 뒤 요청은 실패(`error`)로 남고 요금이 나가지 않는다. 충전한 뒤 다시 실행하면 실패한 기사만 다시 제출된다. 제출 자체가 거절되면 오류를 보여 주고 멈춘다.

In [ ]:
# ── 배치(RUNNER='batch'): 결과 받기 → 남은 기사 계산 → 대기열 자리만큼 제출 ──
if RUNNER == 'batch':
    # 1) 결과 받기
    log = batch_log()
    new_rows = []
    for i, b in log[log['collected'] != 'yes'].iterrows():
        info = openai_client().batches.retrieve(b['batch_id'])
        log.loc[i, 'status'] = info.status
        print(f"{b['batch_id']}: {info.status} / 완료 {info.request_counts.completed} / 실패 {info.request_counts.failed} / 전체 {info.request_counts.total}")
        if info.status not in ('completed', 'failed', 'expired', 'cancelled'):
            continue
        for file_id in [info.output_file_id, info.error_file_id]:
            if not file_id:
                continue
            for line in openai_client().files.content(file_id).text.splitlines():
                if not line.strip():
                    continue
                item = json.loads(line)
                # custom_id = 기사ID|입력 해시
                aid, ihash = item['custom_id'].split('|')
                body = (item.get('response') or {}).get('body') or {}
                now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
                row = dict(article_id=aid, input_hash=ihash, prompt_version=b['prompt_version'], prompt_hash=b['prompt_hash'],
                           runner='batch', model=b['model'], effort=b['effort'], run_utc=now.isoformat(),
                           raw_file=f"{aid[:12]}_{now.strftime('%Y%m%dT%H%M%S')}_batch.json", status='', error='')
                (RAW / row['raw_file']).write_text(json.dumps(item, ensure_ascii=False, indent=1), encoding='utf-8')
                if item.get('error') or not body:
                    row.update(status='error', error=mask(json.dumps(item.get('error') or item.get('response')))[:300])
                else:
                    row = finish(row, body)
                new_rows.append(row)
        if info.status == 'failed' and info.errors:
            print('  배치 실패:', mask(info.errors)[:300])
        log.loc[i, 'collected'] = 'yes'
    if new_rows:
        pd.DataFrame(new_rows).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(),
                                                                     index=False, encoding='utf-8-sig')
        got = pd.DataFrame(new_rows)
        print(f"  결과 {len(got)}건 저장: 성공 {(got['status'] == 'ok').sum()}, 실패 {(got['status'] != 'ok').sum()},"
              f" 비용 ${pd.to_numeric(got.get('cost_usd'), errors='coerce').sum():.2f}")
    if len(log):
        log.to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')
    runs = pd.read_csv(RUNS_PATH, dtype=str).reindex(columns=RUN_COLUMNS)

    # 2) 남은 기사 다시 계산, 3) 대기열 자리만큼 고르기
    todo, waiting = remaining()
    log = batch_log()
    queued = pd.to_numeric(log.loc[log['collected'] != 'yes', 'est_tokens'], errors='coerce').fillna(0).sum()
    room = BATCH_QUEUE_TOKENS * BATCH_FILL - queued
    pick, used = [], 0
    for a in todo:
        if used + est_in(a) > room:
            break
        pick.append(a)
        used += est_in(a)
    cost = api_cost(MODEL, used, PROMPT_TOKENS * len(pick), sum(est_out(a) for a in pick), batch=True)
    print(f'남은 기사 {len(todo):,}건 / 기다리는 배치 약 {queued / 1e6:.2f}백만 토큰 / 넣을 수 있는 자리 약 {max(room, 0) / 1e6:.2f}백만 토큰'
          f' → 이번 제출 {len(pick)}건(약 {used / 1e6:.2f}백만 토큰, 예상 약 ${cost:.2f})')
    spent = pd.to_numeric(runs.loc[runs['runner'] == 'batch', 'cost_usd'], errors='coerce').sum()
    print(f'지금까지 배치 비용 ${spent:.2f}')

    # 4) 제출
    if RUN and pick:
        BATCH_DIR.mkdir(exist_ok=True)
        now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
        name = f"{PROMPT_VERSION}_{MODEL}_{now.strftime('%Y%m%dT%H%M%S')}.jsonl"
        lines = [json.dumps({'custom_id': f'{a}|{text_hash(TEXT[a])}', 'method': 'POST', 'url': '/v1/responses',
                             'body': api_body(TEXT[a])}, ensure_ascii=False) for a in pick]
        (BATCH_DIR / name).write_text('\n'.join(lines) + '\n', encoding='utf-8')
        try:
            client = openai_client()
            with open(BATCH_DIR / name, 'rb') as f:
                uploaded = client.files.create(file=f, purpose='batch')
            batch = client.batches.create(input_file_id=uploaded.id, endpoint='/v1/responses', completion_window='24h',
                                          metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL})
        except Exception as e:          # 크레딧 부족, 한도 초과 등
            (BATCH_DIR / name).unlink(missing_ok=True)
            print('배치 제출 실패:', mask(e)[:300])
        else:
            log = pd.concat([log, pd.DataFrame([dict(batch_id=batch.id, created_utc=now.isoformat(), input_file=name, n=str(len(pick)),
                             est_tokens=str(int(used)), prompt_version=PROMPT_VERSION, prompt_hash=PROMPT_HASH, model=MODEL,
                             effort=EFFORT, status=batch.status, collected='')])])
            log.to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')
            print(f'배치 제출: {batch.id} / {len(pick)}건 / 요청 파일 {name}')
    elif not RUN:
        print('RUN = False: 제출하지 않았다.')

## 6. 점검

이 지시문 버전·모델로 성공한 대상 기사마다 본다(Codex·배치 결과 모두).

| 점검 | 내용 |
|---|---|
| 형식 | 스키마 오류 수 |
| 근거 | 근거 문장이 입력 글에 그대로 있는가(공백 차이 무시). 없는 근거 번호를 가리키는 관계·사건 |
| 참조 | 관계·사건이 없는 개체 id를 가리키는가 |
| 판단 원칙 | 관계 역할 분포와 "기타" 비율, 관련 없는 기사(`relevant=false`) 수, 국내 기업 개체 수, 언급 정도 분포 |

In [ ]:
# ── 점검: 형식, 근거 문장이 원문에 있는지, 근거 번호·개체 id가 맞는지, 기타 역할 비율 ──
squash = lambda s: re.sub(r'\s+', '', s or '')
KOREA_RE = re.compile(r'samsung|sk hynix|hyundai|\blg\b|kia|posco|hanwha|doosan|lotte|korea', re.I)

def ev_ids(x):
    """관계는 evidence_id 하나, 사건은 v3부터 evidence_ids 목록."""
    return x['evidence_ids'] if 'evidence_ids' in x else [x['evidence_id']]


ok_runs = runs[(runs['status'] == 'ok') & (runs['prompt_version'] == PROMPT_VERSION) & (runs['model'] == MODEL)
               & runs['article_id'].isin(wanted)]
ok_runs = ok_runs.drop_duplicates('article_id', keep='last')
raw_dir = lambda runner: EXTRACT / 'raw' / PROMPT_VERSION / (f'api-{MODEL}' if runner == 'batch' else f'codex-{MODEL}')
results, checks = {}, []
for r in ok_runs.itertuples():
    out = parse_result(json.loads((raw_dir(r.runner) / r.raw_file).read_text(encoding='utf-8')))
    results[r.article_id] = out
    src = squash(TEXT[r.article_id])
    ev = {e['id']: e['text'] for e in out['evidence']}
    ids = {e['id'] for e in out['entities']}
    refs = [x['subject'] for x in out['relations']] + [x['object'] for x in out['relations']] + \
           [x['product'] for x in out['relations'] if x['product']] + [i for x in out['events'] for i in x['entities']]
    checks.append(dict(
        article_id=r.article_id, relevant=out['relevant'], type=out['article_type'], schema_errors=sum(1 for _ in VALIDATOR.iter_errors(out)),
        entities=len(out['entities']), relations=len(out['relations']), events=len(out['events']),
        evidence=len(ev), evidence_missing=sum(squash(t) not in src for t in ev.values()),
        bad_evidence_id=sum(i not in ev for x in out['relations'] + out['events'] for i in ev_ids(x)),
        bad_entity_ref=sum(i not in ids for i in refs),
        other_role=sum(x['role'] == '기타' for x in out['relations']),
        korean=sum(bool(KOREA_RE.search(e['name_en'] + ' ' + e['name'])) and e['type'] == '기업' for e in out['entities']),
        runner=r.runner, output_tokens=r.output_tokens))

check = pd.DataFrame(checks)
if len(check):
    check = check.merge(art[['article_id', 'feed_sector', 'source', 'title']], on='article_id')
    check.to_csv(EXTRACT / f'check_{PROMPT_VERSION}_{MODEL}.csv', index=False, encoding='utf-8-sig')
    print(f'점검한 기사 {len(check)}건 / 형식 오류 {check["schema_errors"].sum()} / 원문에 없는 근거 {check["evidence_missing"].sum()}'
          f' / 없는 근거 번호 {check["bad_evidence_id"].sum()} / 없는 개체 id {check["bad_entity_ref"].sum()}')
    print(f'관련 없음 {(~check["relevant"]).sum()}건 / 관계 {check["relations"].sum()}개 중 기타 {check["other_role"].sum()}개'
          f' / 국내 기업 개체 {check["korean"].sum()}개')
    roles = pd.Series([x['role'] for o in results.values() for x in o['relations']]).value_counts()
    mentions = pd.Series([e['mention'] for o in results.values() for e in o['entities']]).value_counts()
    print('관계 역할:', roles.to_dict())
    print('언급 정도:', mentions.to_dict())
check

## 7. 기사 하나 보기

In [ ]:
# ── 기사 하나의 결과를 표로 본다 ──
SHOW = list(sample['article_id'])[0]
if SHOW in results:
    out = results[SHOW]
    names = {e['id']: e['name_en'] or e['name'] for e in out['entities']}
    print(TEXT[SHOW][:300], '…\n')
    print('relevant', out['relevant'], '/', out['article_type'], '/', out['sectors'], '/', out['topics'])
    display(pd.DataFrame(out['entities'])[['id', 'name', 'name_en', 'type', 'category', 'mention', 'search_terms', 'country']])
    if out['relations']:
        rel = pd.DataFrame(out['relations'])
        for k in ('subject', 'object', 'product'):
            rel[k] = rel[k].map(lambda i: names.get(i, i))
        display(rel[['subject', 'role', 'object', 'product', 'status', 'certainty', 'amount_text', 'role_note']])
    if out['events']:
        display(pd.DataFrame(out['events'])[['kind', 'description_ko', 'direction', 'date_text']])
    print('notes:', out['notes'])
else:
    print(SHOW[:12], '결과가 아직 없다.')